# Daniel Kalo, Pranjal Rane
# CS 6120
# Summer 2024
# Final Project - Development of a Clinical Chatbot for Patient Support Using NLP

# Step 1: Collecting and Preprocessing Data from Datasets

In [37]:
import os
import pandas as pd
import xml.etree.ElementTree as ET
import nltk
import json
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/danielkalo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/danielkalo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/danielkalo/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## MedQuAD Data Collecting and Preprocessing

In [38]:
# Function to parse XML files and extract data
def parse_xml(file_path):
    """
    Parses an XML file and extracts question-answer pairs.

    Args:
        file_path (str): The path to the XML file.

    Returns:
        pd.DataFrame: A DataFrame containing the extracted question-answer pairs.
    """
    tree = ET.parse(file_path)
    root = tree.getroot()
    data = []
    for child in root:
        if child.tag == 'QAPairs':
            for qa_pair in child:
                question_element = None
                answer_element = None
                for qa_subchild in qa_pair:
                    if qa_subchild.tag == 'Question':
                        question_element = qa_subchild
                    elif qa_subchild.tag == 'Answer':
                        answer_element = qa_subchild
                
                if question_element is not None and answer_element is not None:
                    question = question_element.text.strip() if question_element.text else ''
                    answer = answer_element.text.strip() if answer_element.text else ''
                    data.append({'question': question, 'answer': answer})
    
    if not data:
        print(f"No data found in file: {file_path}")
    return pd.DataFrame(data)

# Function to preprocess text
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Preprocesses a given text by tokenizing, removing stopwords, and lemmatizing.

    Args:
        text (str): The text to preprocess.

    Returns:
        str: The preprocessed text.
    """
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word.lower()) for word in tokens if word.isalpha() and word.lower() not in stop_words]
    return ' '.join(tokens)

In [42]:
base_dir = 'MedQuAD'
directories = ['1_CancerGov_QA', '2_GARD_QA', '3_GHR_QA', '5_NIDDK_QA', '6_NINDS_QA', 
               '7_SeniorHealth_QA', '8_NHLBI_QA_XML', '9_CDC_QA', '10_MPlus_ADAM_QA', '11_MPlusDrugs_QA', '12_MPlusHerbsSupplements_QA']

data_frames = []
for dir_name in directories:
    dir_path = os.path.join(base_dir, dir_name)
    for file_name in os.listdir(dir_path):
        if file_name.endswith('.xml'):
            file_path = os.path.join(dir_path, file_name)
            df = parse_xml(file_path)
            if not df.empty:
                data_frames.append(df)

# Check if any data frames were collected
if not data_frames:
    raise ValueError("No valid data found in any XML files.")

combined_df = pd.concat(data_frames, ignore_index=True)

# Preprocess the questions and answers
combined_df['preprocessed_question'] = combined_df['question'].apply(preprocess_text)
combined_df['preprocessed_answer'] = combined_df['answer'].apply(preprocess_text)

combined_df.to_csv('MedQuAD_preprocessed.csv', index=False)

No data found in file: MedQuAD/5_NIDDK_QA/0000175.xml
No data found in file: MedQuAD/5_NIDDK_QA/0000177.xml
No data found in file: MedQuAD/5_NIDDK_QA/0000077.xml
No data found in file: MedQuAD/5_NIDDK_QA/0000064.xml
No data found in file: MedQuAD/5_NIDDK_QA/0000065.xml
No data found in file: MedQuAD/5_NIDDK_QA/0000056.xml
No data found in file: MedQuAD/6_NINDS_QA/0000007.xml
No data found in file: MedQuAD/6_NINDS_QA/0000244.xml
No data found in file: MedQuAD/6_NINDS_QA/0000182.xml
No data found in file: MedQuAD/6_NINDS_QA/0000018.xml


In [43]:
import re

# Load the preprocessed data
combined_df = pd.read_csv('MedQuAD_preprocessed.csv')

# Ensure NLTK resources are available
nltk.download('punkt')

# Function to normalize text
def normalize_text(text):
    """
    Normalizes a given text by removing special characters and converting to lowercase.

    Args:
        text (str): The text to normalize.

    Returns:
        str: The normalized text.
    """
    if isinstance(text, float):
        text = ""  # Handle NaN values
    # Remove special characters
    text = re.sub(r'\W', ' ', text)
    # Convert to lowercase
    text = text.lower()
    return text

# Additional preprocessing steps
combined_df.dropna(subset=['question', 'answer'], inplace=True)
combined_df.drop_duplicates(subset=['question', 'answer'], inplace=True)

# Ensure all entries are strings
combined_df['preprocessed_question'] = combined_df['preprocessed_question'].astype(str)
combined_df['preprocessed_answer'] = combined_df['preprocessed_answer'].astype(str)

combined_df['normalized_question'] = combined_df['preprocessed_question'].apply(normalize_text)
combined_df['normalized_answer'] = combined_df['preprocessed_answer'].apply(normalize_text)
combined_df['tokenized_question'] = combined_df['normalized_question'].apply(word_tokenize)
combined_df['tokenized_answer'] = combined_df['normalized_answer'].apply(word_tokenize)

# Save the final preprocessed data
combined_df.to_csv('MedQuAD_final_preprocessed.csv', index=False)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/danielkalo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## COVID-QA Data Collecting and Preprocessing

In [44]:
# Load the dataset
with open('COVID-QA/data/question-answering/COVID-QA.json') as f:
    data = json.load(f)

# Extract question-answer pairs based on the inspected structure
qa_pairs = []
for entry in data['data']:  # Accessing 'data' key
    for paragraph in entry['paragraphs']:
        for qa in paragraph['qas']:
            question = qa['question']
            answer = qa['answers'][0]['text'] if qa['answers'] else ''
            qa_pairs.append({'question': question, 'answer': answer})

# Convert to DataFrame
df = pd.DataFrame(qa_pairs)

# Additional preprocessing steps
df.dropna(subset=['question', 'answer'], inplace=True)
df.drop_duplicates(subset=['question', 'answer'], inplace=True)

# Ensure all entries are strings
df['question'] = df['question'].astype(str)
df['answer'] = df['answer'].astype(str)

df['normalized_question'] = df['question'].apply(normalize_text)
df['normalized_answer'] = df['answer'].apply(normalize_text)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

df['tokenized_question'] = df['normalized_question'].apply(preprocess_text)
df['tokenized_answer'] = df['normalized_answer'].apply(preprocess_text)

# Save the preprocessed data
df.to_csv('COVID_QA_preprocessed.csv', index=False)

In [45]:
# Load the COVID-QA preprocessed data
covid_qa_df = pd.read_csv('COVID_QA_preprocessed.csv')

# Function to preprocess text (same as used for MedQuAD)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Add preprocessed_question and preprocessed_answer columns
covid_qa_df['preprocessed_question'] = covid_qa_df['question'].apply(preprocess_text)
covid_qa_df['preprocessed_answer'] = covid_qa_df['answer'].apply(preprocess_text)

# Save the updated COVID-QA dataset
covid_qa_df.to_csv('COVID_QA_final_preprocessed.csv', index=False)